In [1]:
import os
#os.chdir(os.path.expanduser("ccFBApy"))
import cfba_moment as cfba

In [2]:
#!pip install cobra

# Load model

In [3]:
import cobra
modelorg = cobra.io.read_sbml_model('models\\iAF1260.xml')

In [4]:
import pandas as pd
mw=pd.read_csv(r"data\ec_mw.csv")
kcat=pd.read_csv(r"data\ec_kcat.csv")

Res=cfba.cfba_moment(modelorg,kcat,MW=mw,verboseMode=1,RHS=0.27,medval=3600*22.6)
print("Done...")

Preparing LP ...
Adding variables...
Adding constraints...
Solving LP ...
Done...


In [5]:
print(Res['obj'])

0.4452932136236324


In [6]:
CSList=['EX_glc__D_e', 'EX_glyc_e', 'EX_ac_e', 'EX_fru_e', 'EX_pyr_e', 'EX_gal_e', 'EX_lac__L_e', 'EX_malt_e', 'EX_mal__L_e', 'EX_fum_e', 'EX_xyl__D_e', 'EX_man_e', 'EX_tre_e', 'EX_mnl_e', 'EX_g6p_e', 'EX_succ_e', 'EX_gam_e', 'EX_sbt__D_e', 'EX_glcn_e', 'EX_rib__D_e', 'EX_gsn_e', 'EX_ala__L_e', 'EX_akg_e', 'EX_acgam_e']
msrd=[0.66,0.47,0.29,0.54,0.41,0.24,0.41,0.52,0.55,0.47,0.51,0.35,0.48,0.61,0.78,0.50,0.40,0.48,0.68,0.41,0.37,0.24,0.24,0.61]
CA=[6,3,2,6,3,6,3,12,4,4,5,6,12,6,6,4,6,6,6,5,10,3,5,8]

In [7]:
for rid in CSList:
    modelorg.reactions.get_by_id(rid).lower_bound=0
    modelorg.reactions.get_by_id(rid).upper_bound=0
    #cL.append(rid[2:-3])
modelorg.reactions.get_by_id('EX_o2_e').lower_bound=-1000.0
modelorg.reactions.get_by_id('ATPM').lower_bound=0
modelorg.reactions.get_by_id('ATPM').upper_bound=0

In [8]:
modelR=modelorg.copy()

# Test MOMENT model

In [9]:
C_mr=0.27
allRes=[]
bmv=[]
for cs in range(24):
#for cs in [1]:    
    print(CSList[cs])
    model=modelR.copy()
    model.reactions.get_by_id(CSList[cs]).lower_bound=-1000.0
    Res=cfba.cfba_moment(model,kcat,MW=mw,verboseMode=3,RHS=0.27,medval=3600*22.6)
    bmv.append(Res['obj'])
    print(Res['obj'],Res['status'])
    del model
    del Res

EX_glc__D_e
Preparing LP ...
Process: 2AGPA140tipp b2835 1 -1
Process: 2AGPA141tipp b2835 3 -1
Process: 2AGPA160tipp b2835 4 -1
Process: 3OAR180 b1093 5 5
Process: 2AGPA161tipp b2835 6 -1
Process: 3OAR181 b1093 7 -1
Process: 2AGPA180tipp b2835 8 -1
Process: 3OAR40 b1093 9 9
Process: 2AGPA181tipp b2835 10 -1
Process: 3OAR60 b1093 11 11
Process: 3OAR80 b1093 12 12
Process: 3OAS100 b1095 or b2323 13 -1
Process: 2AGPE120tipp b2835 14 -1
Process: 3OAS120 b1095 or b2323 15 -1
Process: 3OAS121 b2323 16 -1
Process: 3OAS140 b2323 or b1095 17 -1
Process: 3OAS141 b2323 20 -1
Process: 2AGPE140tipp b2835 22 -1
Process: 3OAS160 b1095 or b2323 25 -1
Process: 3OAS161 b2323 26 -1
Process: 3OAS180 b2323 or b1095 27 -1
Process: 3OAS181 b1095 28 -1
Process: 3OAS60 b2323 or b1095 29 -1
Process: 3OAS80 b2323 or b1095 30 -1
Process: 3PEPTabcpp b1245 and b1329 and b1246 and b1247 and b1244 31 -1
Rxn:3PEPTabcpp gpr: b1245 and b1329 and b1246 and b1247 and b1244, straight ANDs
Process: 3PEPTtex b0929 or b1377 o

## Running time

In [10]:
import time
print(time.time())
print(bmv)

1794738618.8350081
[0.49252495939851, 0.5260897983669187, 0.316459006756192, 0.47871378287602984, 0.39105794132445626, 0.4496186332277674, 0.3836926496702989, 0.48350147630558427, 0.40483244087999665, 0.4040350387053724, 0.4666734012949789, 0.46308700147494836, 0.5078274088579118, 0.4454903544873947, 0.560647095058716, 0.41646480244390854, 0.46906257264659296, 0.46593995663182514, 0.4882040075822601, 0.39015490237158096, 0.5666989833278767, 0.3930005939873006, 0.443744214937267, 0.4248493287327286]


In [11]:
from scipy.stats import pearsonr
corr, _ = pearsonr(bmv, msrd)
print(corr)

0.4267808290674814
